In [1]:
import os
import pickle

import pandas as pd

In [59]:
def get_median_runs(df, metric="l2", bench="function", pow_basis=None, pow_res=None):
    """
    For each (method, function, G, width, depth) group, keep only the run whose
    `metric` value is closest to the group's median.

    If pow_basis and pow_res are given, only keep that specific variant of the
    'power' method (plus all other methods).
    """
    df = df.copy()

    # Optional: restrict to a specific (pow_basis, pow_res) for the power method
    if pow_basis is not None and pow_res is not None:
        df = df[
            (df["method"] != "power") |
            ((df["pow_basis"] == pow_basis) & (df["pow_res"] == pow_res))
        ]

    # Drop rows without metric (can't define a median on them)
    df = df.dropna(subset=[metric])

    # Define what "same experiment with multiple runs" means:
    group_cols = ["method", bench, "G", "width", "depth"]
    # (If you really want pow_basis/pow_res to distinguish architectures *even without*
    # filtering, you can change this to:
    # group_cols = ["method", "function", "G", "width", "depth", "pow_basis", "pow_res"]
    # but make sure pow_basis/pow_res are not NaN for non-power methods.)

    def idx_of_median(s: pd.Series):
        """Return index of the row whose metric is closest to the median."""
        med = s.median()
        return (s - med).abs().idxmin()

    # For each group, find the index of the "median" run
    idx = df.groupby(group_cols)[metric].apply(idx_of_median)

    # idx is a Series of original row indices; select those rows
    median_df = df.loc[idx.values].reset_index(drop=True)

    return median_df


def count_pow_better_than_baseline(
    best_df,
    bench="function",      # or "pde"
    metric="l2",           # used when both_metrics=False
    both_metrics=False     # when True → require power better on BOTH l2 and loss
):
    # Split pow and baseline
    pow_df = best_df[best_df["method"] == "power"].copy()
    base_df = best_df[best_df["method"] == "baseline"].copy()

    # Define how we align architectures between pow and baseline
    merge_cols = [bench, "G", "width", "depth"]

    if both_metrics:
        # We need both l2 and loss side by side
        metrics = ["l2", "loss"]
        merged = pow_df.merge(
            base_df[merge_cols + metrics],
            on=merge_cols,
            suffixes=("_pow", "_baseline"),
        )

        # power must be better on BOTH l2 and loss
        better_l2   = merged["l2_pow"]   < merged["l2_baseline"]
        better_loss = merged["loss_pow"] < merged["loss_baseline"]
        better_mask = better_l2 & better_loss

    else:
        # Original behavior: single metric
        merged = pow_df.merge(
            base_df[merge_cols + [metric]],
            on=merge_cols,
            suffixes=("_pow", "_baseline"),
        )

        better_mask = merged[f"{metric}_pow"] < merged[f"{metric}_baseline"]

    # Count per bench (function or pde) how many times power wins
    pow_better_count = better_mask.groupby(merged[bench]).sum()
    pow_better_count.name = "pow_better_count"

    # Total # of comparable architectures per bench
    total_arch = merged.groupby(bench).size()
    total_arch.name = "total_architectures"

    summary = pd.concat([pow_better_count, total_arch], axis=1)
    summary["pow_better_ratio"] = 100 * summary["pow_better_count"] / summary["total_architectures"]

    return summary, merged


def evaluate_pow_configs_median(df, bench="function", metric="l2"):
    """
    For each (pow_basis, pow_res) used by method 'power', compute how often
    power has a LOWER median-run `metric` than baseline on the same
    (function, G, width, depth) architectures.

    Returns:
        summary_df: DataFrame with one row per (pow_basis, pow_res)
        best_row:   The row of summary_df with the highest pow_better_ratio
    """
    df = df.copy()
    df = df.dropna(subset=[metric])

    # All distinct pow parameter pairs for method 'power'
    pow_params = (
        df.loc[df["method"] == "power", ["pow_basis", "pow_res"]]
          .dropna()
          .drop_duplicates()
    )

    results = []

    for _, p in pow_params.iterrows():
        basis = p["pow_basis"]
        res   = p["pow_res"]

        # Get median runs for this specific power variant + all other methods
        median_df = get_median_runs(df, metric=metric,
                                    pow_basis=basis, pow_res=res, bench=bench)

        pow_med  = median_df[median_df["method"] == "power"].copy()
        base_med = median_df[median_df["method"] == "baseline"].copy()

        if pow_med.empty or base_med.empty:
            results.append({
                "pow_basis": basis,
                "pow_res": res,
                "pow_better_count": 0,
                "total_arch": 0,
                "pow_better_ratio": np.nan,
            })
            continue

        # architectures we compare on
        arch_cols = [bench, "G", "width", "depth"]

        merged = pow_med.merge(
            base_med[arch_cols + [metric]],
            on=arch_cols,
            suffixes=("_pow", "_baseline")
        )

        total_arch = len(merged)
        if total_arch == 0:
            pow_better = 0
            ratio = np.nan
        else:
            better_mask = merged[f"{metric}_pow"] < merged[f"{metric}_baseline"]
            pow_better = better_mask.sum()
            ratio = 100 * pow_better / total_arch

        results.append({
            "pow_basis": basis,
            "pow_res": res,
            "pow_better_count": pow_better,
            "total_arch": total_arch,
            "pow_better_ratio": ratio,
        })

    summary_df = (
        pd.DataFrame(results)
          .sort_values("pow_better_ratio", ascending=False)
          .reset_index(drop=True)
    )

    best_row = summary_df.iloc[0] if not summary_df.empty else None
    return summary_df, best_row

In [46]:
filedir = "ff_results/grid_search.csv"

df = pd.read_csv(filedir)

In [47]:
summary, best = evaluate_pow_configs_median(df, bench="function", metric="loss")

print(summary[:20])  # all (pow_basis, pow_res) combos with stats
print("\nBest configuration:\n", best)

    pow_basis  pow_res  pow_better_count  total_arch  pow_better_ratio
0        1.00     0.25               462         480          0.962500
1        1.50     0.25               461         480          0.960417
2        2.00     0.25               460         480          0.958333
3        1.25     0.25               460         480          0.958333
4        1.75     0.25               459         480          0.956250
5        0.75     0.50               449         480          0.935417
6        0.75     1.00               447         480          0.931250
7        0.75     0.25               446         480          0.929167
8        1.75     0.50               446         480          0.929167
9        1.00     0.50               446         480          0.929167
10       2.00     0.50               446         480          0.929167
11       1.50     0.50               445         480          0.927083
12       1.00     0.75               445         480          0.927083
13    

In [48]:
med_df = get_median_runs(df, pow_basis=1.00, pow_res=0.25, metric="loss", bench="function")

In [49]:
summary, merged_pairs = count_pow_better_than_baseline(med_df, metric="loss", bench="function", both_metrics=True)
print(summary)

          pow_better_count  total_architectures  pow_better_ratio
function                                                         
f1                      93                   96         96.875000
f2                      92                   96         95.833333
f3                      94                   96         97.916667
f4                      84                   96         87.500000
f5                      86                   96         89.583333


In [62]:
filedir = "pde_results/grid_search.csv"

df = pd.read_csv(filedir)

In [43]:
summary, best = evaluate_pow_configs_median(df, bench="pde", metric="loss")

print(summary[:20])  # all (pow_basis, pow_res) combos with stats
print("\nBest configuration:\n", best)

    pow_basis  pow_res  pow_better_count  total_arch  pow_better_ratio
0        0.75     0.50               172         215          0.800000
1        0.75     0.25               170         215          0.790698
2        1.00     0.25               170         215          0.790698
3        0.75     1.50               167         215          0.776744
4        0.75     1.75               166         215          0.772093
5        1.75     0.25               166         215          0.772093
6        1.00     1.00               166         215          0.772093
7        0.75     2.00               166         215          0.772093
8        0.75     0.75               164         215          0.762791
9        2.00     0.25               163         216          0.754630
10       1.00     1.50               162         215          0.753488
11       1.25     0.25               162         215          0.753488
12       0.75     1.25               161         215          0.748837
13    

In [64]:
med_df = get_median_runs(df, pow_basis=1.00, pow_res=0.25, metric="loss", bench="pde")

In [65]:
summary, merged_pairs = count_pow_better_than_baseline(med_df, metric="loss", bench="pde", both_metrics=True)
print(summary)

           pow_better_count  total_architectures  pow_better_ratio
pde                                                               
ac                       60                   72         83.333333
burgers                  39                   71         54.929577
helmholtz                43                   72         59.722222
